# Classes and Object-Oriented Programming in Python

Object-oriented programming (OOP) organizes code around **objects** that combine:

- **data**: the state of an object;
- **behavior**: operations the object can perform.

In this notebook we start from the basic ideas and progressively move toward patterns that appear in real Python and machine-learning code.

## Learning objectives

By the end of this notebook, you should be able to:

- define a class and create objects;
- understand `self`, attributes, methods, and constructors;
- distinguish instance attributes from class attributes;
- use properties for computed or validated attributes;
- understand inheritance, method overriding, and polymorphism;
- understand why composition is often preferable to inheritance;
- use `@classmethod`, `@staticmethod`, and `@dataclass`;
- recognize useful special methods such as `__repr__` and `__len__`;
- understand abstract base classes;
- recognize OOP patterns used by machine-learning libraries;
- define and train a small neural network as a Python class.

# Part I — The basic idea

## 1. From data to objects

A dictionary can represent an entity such as a patient:

In [ ]:
patient = {
    "name": "Anna",
    "age": 42,
    "temperature": 37.1
}

print(patient["name"])

A dictionary stores data, but it does not naturally define the **behavior** associated with that data.

A class lets us define a new type of object.

In [ ]:
class Patient:
    pass


p = Patient()

print(p)
print(type(p))

`Patient` is the **class**.

`p` is an **instance** (an object created from that class).

```text
Class = blueprint
        ↓
Patient
        ↓
   ┌────┴────┐
   ↓         ↓
 Anna       Marco
 objects / instances
```

## 2. The constructor: `__init__`

The method `__init__` is called when a new object is created.

It is commonly used to initialize the object's attributes.

In [ ]:
class Patient:
    def __init__(self, name, age, temperature):
        self.name = name
        self.age = age
        self.temperature = temperature


patient = Patient("Anna", 42, 37.1)

print(patient.name)
print(patient.age)
print(patient.temperature)

## 3. What is `self`?

`self` refers to the **current object**.

When we create:

```python
anna = Patient("Anna", 42, 37.1)
marco = Patient("Marco", 67, 38.4)
```

each object has its own values:

```text
anna.name  → "Anna"
marco.name → "Marco"
```

Inside the class, `self.name` means:

> the `name` attribute of this particular object.

In [ ]:
anna = Patient("Anna", 42, 37.1)
marco = Patient("Marco", 67, 38.4)

print(anna.name)
print(marco.name)

## 4. Methods: objects can have behavior

A **method** is a function defined inside a class.

Here the patient object can determine whether it has a fever.

In [ ]:
class Patient:
    def __init__(self, name, age, temperature):
        self.name = name
        self.age = age
        self.temperature = temperature

    def has_fever(self, threshold=38.0):
        return self.temperature >= threshold


anna = Patient("Anna", 42, 37.0)
marco = Patient("Marco", 67, 38.4)

print("Anna:", anna.has_fever())
print("Marco:", marco.has_fever())

Notice the call:

```python
marco.has_fever()
```

Conceptually, Python passes `marco` as `self`.

The method can therefore access the state stored inside that object.

## 5. State can change

Objects can maintain state over time.

In [ ]:
patient = Patient("Anna", 42, 37.1)

print("Before:", patient.temperature)

patient.temperature = 38.3

print("After:", patient.temperature)
print("Has fever:", patient.has_fever())

# Part II — Making classes more useful

## 6. `__repr__`: a useful representation

Special methods surrounded by double underscores are often called **dunder methods**.

`__repr__` controls how an object is represented, which is especially useful for debugging and notebooks.

In [ ]:
class Patient:
    def __init__(self, name, age, temperature):
        self.name = name
        self.age = age
        self.temperature = temperature

    def has_fever(self, threshold=38.0):
        return self.temperature >= threshold

    def __repr__(self):
        return (
            f"Patient(name={self.name!r}, "
            f"age={self.age}, "
            f"temperature={self.temperature})"
        )


patient = Patient("Anna", 42, 37.1)
patient

## 7. Encapsulation

A class groups related **data and behavior** together.

For example, a rectangle stores its dimensions and knows how to compute quantities derived from them.

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


rectangle = Rectangle(4, 3)

print("Area:", rectangle.area())
print("Perimeter:", rectangle.perimeter())

## 8. Properties

A property lets us expose a method as if it were an attribute.

This is useful for **computed values**.

In [ ]:
import math


class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        return math.pi * self.radius**2


circle = Circle(2)

print(circle.radius)
print(circle.area)      # no parentheses

Properties are also useful when we want to **validate assignments**.

Here we prevent a negative temperature in Kelvin.

In [ ]:
class Sample:
    def __init__(self, temperature):
        self.temperature = temperature

    @property
    def temperature(self):
        return self._temperature

    @temperature.setter
    def temperature(self, value):
        if value < 0:
            raise ValueError("Temperature in Kelvin cannot be negative.")
        self._temperature = value


sample = Sample(300)
print(sample.temperature)

# Try:
# sample.temperature = -10

The underscore in `_temperature` is a Python convention meaning:

> this attribute is intended for internal use.

It is not truly private, but users of the class are encouraged to interact through the public interface `temperature`.

## 9. Class attributes vs instance attributes

An **instance attribute** belongs to one object.

A **class attribute** is shared conceptually by all objects of the class.

In [ ]:
class Patient:
    fever_threshold = 38.0  # class attribute

    def __init__(self, name, temperature):
        self.name = name              # instance attribute
        self.temperature = temperature

    def has_fever(self):
        return self.temperature >= self.fever_threshold


anna = Patient("Anna", 37.5)
marco = Patient("Marco", 38.3)

print(anna.has_fever())
print(marco.has_fever())
print(Patient.fever_threshold)

# Part III — More advanced class features

## 10. `@classmethod`

A class method receives the **class** itself (`cls`) rather than an individual object.

A common use is to define an **alternative constructor**.

In [ ]:
class Patient:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_birth_year(cls, name, birth_year, current_year=2026):
        age = current_year - birth_year
        return cls(name, age)


anna = Patient.from_birth_year("Anna", 1984)

print(anna.name)
print(anna.age)

## 11. `@staticmethod`

A static method belongs conceptually to the class, but it does not need either `self` or `cls`.

It behaves like a normal function grouped inside the class namespace.

In [ ]:
class Temperature:
    @staticmethod
    def celsius_to_fahrenheit(celsius):
        return celsius * 9 / 5 + 32


print(Temperature.celsius_to_fahrenheit(37))

A useful mental model:

```text
instance method   → needs an object        → self
class method      → needs the class        → cls
static method     → needs neither
```

## 12. Dataclasses

Many classes mainly store data.

Python's `@dataclass` decorator can automatically generate methods such as:

- `__init__`
- `__repr__`
- `__eq__`

This removes repetitive boilerplate.

In [ ]:
from dataclasses import dataclass


@dataclass
class PatientRecord:
    name: str
    age: int
    temperature: float


anna = PatientRecord("Anna", 42, 37.1)
marco = PatientRecord("Marco", 67, 38.4)

print(anna)
print(anna == marco)

Type annotations such as:

```python
name: str
age: int
```

document the intended types.

Python does **not** automatically enforce them at runtime, but they help readers, IDEs, and static type checkers.

## 13. Other useful dunder methods

Classes can behave like built-in Python objects by implementing special methods.

For example, `__len__` lets us use `len(object)`.

In [ ]:
class MeasurementSeries:
    def __init__(self, values):
        self.values = list(values)

    def __len__(self):
        return len(self.values)

    def __getitem__(self, index):
        return self.values[index]


series = MeasurementSeries([10, 12, 15, 9])

print(len(series))
print(series[0])
print(series[1:3])

Because we implemented `__len__` and `__getitem__`, our custom object starts to behave like a Python container.

# Part IV — Relationships between classes

## 14. Inheritance

Inheritance lets one class reuse and extend another class.

The child class inherits attributes and methods from the parent class.

In [ ]:
class Person:
    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"Person: {self.name}"


class Student(Person):
    def __init__(self, name, student_id):
        super().__init__(name)
        self.student_id = student_id

    def describe(self):
        return f"Student: {self.name}, ID={self.student_id}"


student = Student("Elena", "S001")

print(student.name)
print(student.describe())

`super()` lets the child class reuse behavior from its parent.

`Student.describe()` **overrides** `Person.describe()`.

## 15. Polymorphism

Different classes can provide the same interface.

The calling code does not necessarily need to know the exact class.

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius**2


class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


shapes = [
    Circle(2),
    Rectangle(3, 4),
    Circle(1)
]

for shape in shapes:
    print(type(shape).__name__, "area =", shape.area())

The loop only assumes:

```python
shape.area()
```

It does not care whether `shape` is a `Circle` or a `Rectangle`.

This is **polymorphism**.

## 16. Abstract base classes

Sometimes we want to explicitly define an interface that subclasses must implement.

Python provides **abstract base classes** through the `abc` module.

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):

    @abstractmethod
    def area(self):
        pass


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


rectangle = Rectangle(4, 3)
print(rectangle.area())

# Shape() would fail because Shape is abstract.

This becomes useful in larger software projects because it makes the expected interface explicit.

## 17. Composition

Inheritance models an **is-a** relationship:

```text
Student is a Person
```

Composition models a **has-a** relationship:

```text
Person has an Address
```

Often, composition is more flexible.

In [ ]:
@dataclass
class Address:
    city: str
    country: str


class Person:
    def __init__(self, name, address):
        self.name = name
        self.address = address


address = Address("Trieste", "Italy")
person = Person("Anna", address)

print(person.name)
print(person.address.city)

## Inheritance or composition?

A useful rule of thumb:

```text
"is a"  → inheritance may make sense
"has a" → composition is usually natural
```

Do not use inheritance only to reuse a few lines of code.

# Part V — OOP in machine learning

Modern ML libraries make extensive use of classes.

For example, in PyTorch:

```python
class MyNetwork(nn.Module):
    ...
```

A neural network is naturally represented as an object because it has:

- **state**: its trainable parameters;
- **structure**: its layers;
- **behavior**: the forward computation;
- methods inherited from a framework class such as `.parameters()`, `.train()`, and `.eval()`.

## 18. A neural network as a class

We will build a small neural network for a synthetic binary-classification problem.

If PyTorch is not installed in the active environment, install it before running this section.

```bash
python -m pip install torch matplotlib
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

### Create a simple dataset

Each observation has two features.

The label is `1` when the point is approximately outside a circle and `0` otherwise. This creates a **non-linear classification problem**.

In [ ]:
n_samples = 1000

X = torch.rand(n_samples, 2) * 2 - 1

distance_squared = X[:, 0]**2 + X[:, 1]**2
y = (distance_squared > 0.5).float().reshape(-1, 1)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
plt.figure(figsize=(6, 5))

plt.scatter(
    X[:, 0].numpy(),
    X[:, 1].numpy(),
    c=y[:, 0].numpy(),
    s=15
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Synthetic classification problem")
plt.show()

### Define the neural network

In PyTorch, neural networks inherit from `nn.Module`.

The constructor defines the layers.

The `forward()` method defines how data moves through the network.

In [ ]:
class SimpleClassifier(nn.Module):

    def __init__(self, input_dim=2, hidden_dim=16):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.network(x)


model = SimpleClassifier()

print(model)

This is a good example of several OOP ideas working together:

```text
SimpleClassifier
      │
      └── inherits from nn.Module
                 │
                 ├── parameters()
                 ├── train()
                 ├── eval()
                 └── many other methods

SimpleClassifier also contains other objects:

nn.Linear
nn.ReLU
nn.Sequential
```

So PyTorch uses both **inheritance** and **composition**.

### Inspect the model state

The weights and biases are stored inside the object.

In [ ]:
for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape))

### Train the model

The optimizer receives the parameters belonging to our model object.

`BCEWithLogitsLoss` combines a sigmoid transformation with binary cross-entropy in a numerically stable implementation.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

loss_history = []

for epoch in range(300):

    model.train()

    optimizer.zero_grad()

    logits = model(X)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | loss = {loss.item():.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training loss")
plt.show()

### Evaluate the model

During evaluation we use:

```python
model.eval()
```

and disable gradient computation with:

```python
torch.no_grad()
```

In [ ]:
model.eval()

with torch.no_grad():
    logits = model(X)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).float()

accuracy = (predictions == y).float().mean()

print(f"Training accuracy: {accuracy.item():.3f}")

### Visualize the learned decision boundary

In [ ]:
grid_values = torch.linspace(-1, 1, 200)

xx, yy = torch.meshgrid(
    grid_values,
    grid_values,
    indexing="xy"
)

grid = torch.stack(
    [xx.reshape(-1), yy.reshape(-1)],
    dim=1
)

model.eval()

with torch.no_grad():
    grid_prob = torch.sigmoid(model(grid))

zz = grid_prob.reshape(xx.shape)

plt.figure(figsize=(6, 5))

plt.contourf(
    xx.numpy(),
    yy.numpy(),
    zz.numpy(),
    levels=20,
    alpha=0.6
)

plt.scatter(
    X[:, 0].numpy(),
    X[:, 1].numpy(),
    c=y[:, 0].numpy(),
    s=10
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Decision surface learned by the neural network")
plt.show()

## Why is OOP useful for a neural network?

Imagine defining a model only with unrelated variables and functions:

```text
weights_layer_1
bias_layer_1
weights_layer_2
bias_layer_2
forward_function(...)
...
```

As the model grows, it becomes difficult to manage.

Instead, the class gives us one coherent object:

```python
model = SimpleClassifier()
```

that contains:

```text
model
├── architecture
├── weights
├── biases
├── forward computation
├── train/eval mode
└── inherited framework functionality
```

This is one of the reasons OOP is so common in machine learning libraries.

# Part VI — Exercises

## Exercise 1 — Measurement series

Create a class `MeasurementSeries` with:

- an attribute `values`;
- a method `mean()`;
- a method `minimum()`;
- a method `maximum()`;
- a property `count`;
- a `__repr__` method;
- a `__len__` method.

Then create an object and test all methods.

In [ ]:
# Write your solution here

## Possible solution

In [ ]:
class MeasurementSeries:
    def __init__(self, values):
        self.values = list(values)

    def mean(self):
        return sum(self.values) / len(self.values)

    def minimum(self):
        return min(self.values)

    def maximum(self):
        return max(self.values)

    @property
    def count(self):
        return len(self.values)

    def __len__(self):
        return len(self.values)

    def __repr__(self):
        return f"MeasurementSeries(n={len(self)}, values={self.values})"


series = MeasurementSeries([10, 12, 15, 9])

print(series)
print("Mean:", series.mean())
print("Minimum:", series.minimum())
print("Maximum:", series.maximum())
print("Count:", series.count)
print("len(series):", len(series))

## Exercise 2 — Extend the neural network

Modify `SimpleClassifier` so that:

1. the number of hidden layers can be changed;
2. the hidden dimension can be changed;
3. dropout can optionally be added.

Think about which information should become constructor arguments and which objects should be stored as attributes.

# When should you use classes?

Classes are especially useful when:

- you have many entities with the same structure;
- data and behavior naturally belong together;
- objects maintain state over time;
- several components interact through a common interface;
- you are building reusable software components.

Classes are **not automatically better than functions**.

For a simple transformation such as:

```python
celsius_to_fahrenheit(x)
```

a normal function may be clearer than defining a class.

# Take-home messages

- A **class** defines a new type of object.
- An **object** combines state and behavior.
- `__init__` initializes object state.
- `self` refers to the current instance.
- Methods operate on object state.
- Properties provide controlled or computed attribute access.
- Class methods operate on the class; static methods need neither an instance nor the class.
- `@dataclass` reduces boilerplate for data-oriented classes.
- Dunder methods let custom classes integrate naturally with Python.
- Inheritance expresses an **is-a** relationship.
- Composition expresses a **has-a** relationship and is often more flexible.
- Polymorphism lets different objects expose a common interface.
- Abstract base classes can make an expected interface explicit.
- ML frameworks such as PyTorch use OOP extensively: a neural network object contains its layers, parameters, state, and behavior.